# Setup

In [1]:
!pip install -q transformers gensim

import re
import collections
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Word-Level Tokenization (OOV problem)

In [2]:
review = "This book kept me up all night - totally unputdownable"

#naive word-level tokenization
word_tokens = review.lower().replace("-", "").replace("!", "").split(" ")
print(word_tokens)

['this', 'book', 'kept', 'me', 'up', 'all', 'night', '', 'totally', 'unputdownable']


In [3]:
# how much your model will know,
# this depends on entire dataset.

vocab = {"this": 0, "book": 1, "kept": 2, "me": 3, "up": 4, "all": 5, "night": 6, "totally": 7}
vocab

{'this': 0,
 'book': 1,
 'kept': 2,
 'me': 3,
 'up': 4,
 'all': 5,
 'night': 6,
 'totally': 7}

In [6]:
ids = [vocab.get(tok, -1) for tok in word_tokens] # -1 stands in for <UNK>
print(ids)

[0, 1, 2, 3, 4, 5, 6, -1, 7, -1]


# Example

In [7]:
sentence = "The plot twist blew my mind!"
tokens = sentence.lower().replace("!", "").split()
print(tokens)
print([vocab.get(tok, -1) for tok in tokens])

['the', 'plot', 'twist', 'blew', 'my', 'mind']
[-1, -1, -1, -1, -1, -1]


# Byte-Pair Encoding

In [9]:
from transformers import GPT2Tokenizer

bpe_tokenizer = GPT2Tokenizer.from_pretrained("gpt2") # 2020

sentence = "This book was unputdownable and utterly captivating."

print("BPE (GPT-2):     ", bpe_tokenizer.tokenize(sentence))

BPE (GPT-2):      ['This', 'Ġbook', 'Ġwas', 'Ġun', 'put', 'down', 'able', 'Ġand', 'Ġutterly', 'Ġcapt', 'ivating', '.']


# WordPiece Tokenizer - BERT model

In [11]:
from transformers import BertTokenizer

wordpiece_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

print("WordPiece (BERT): ", wordpiece_tokenizer.tokenize(sentence))

WordPiece (BERT):  ['this', 'book', 'was', 'un', '##put', '##down', '##able', 'and', 'utterly', 'capt', '##ivating', '.']


# Example

In [12]:
sentence2 = "I asked Zylotech for a refund"
print("BPE (GPT-2):     ", bpe_tokenizer.tokenize(sentence2))
print("WordPiece (BERT): ", wordpiece_tokenizer.tokenize(sentence2))

BPE (GPT-2):      ['I', 'Ġasked', 'ĠZ', 'yl', 'otech', 'Ġfor', 'Ġa', 'Ġrefund']
WordPiece (BERT):  ['i', 'asked', 'z', '##yl', '##ote', '##ch', 'for', 'a', 'ref', '##und']


# Word2Vec

In [13]:
from gensim.models import Word2Vec

# Toy corpus of "reviews" (already tokenized into lowercase words)
reviews = [
    "this book kept me up all night it was so good".split(),
    "the plot twist blew my mind absolutely brilliant".split(),
    "boring book i almost fell asleep reading it".split(),
    "the mystery kept me guessing until the very end".split(),
    "the ending was predictable and honestly quite boring".split(),
    "brilliant characters and a gripping plot throughout".split(),
    "i could not put this book down it was gripping".split(),
]

model = Word2Vec(
    sentences=reviews,
    vector_size=50,   # dimensionality of each word vector
    window=2,          # how many neighboring words count as "context"
    min_count=1,        # keep even rare words for this tiny demo
    sg=1,                # sg=1 -> Skip-gram, sg=0 -> CBOW
    epochs=200,
)

In [14]:
model.wv

In [17]:
model.wv['gripping']  #word gripping in 50 dimesions

array([-0.0093056 ,  0.01001705, -0.01113353,  0.00610174,  0.00811732,
        0.009136  ,  0.01068291,  0.02038326, -0.03946117,  0.0056218 ,
       -0.01117492, -0.02531192,  0.03723491,  0.00177458,  0.01055829,
        0.00133532,  0.03547882,  0.04707108, -0.04496187, -0.04835492,
       -0.00505999,  0.00664914,  0.02326781, -0.01265547,  0.02837154,
        0.00847155,  0.01865246,  0.02233727, -0.02587591,  0.00528369,
        0.01105741, -0.03548636, -0.0033032 , -0.01383698, -0.03093695,
        0.00035013,  0.017388  ,  0.00182003,  0.004889  , -0.0104075 ,
        0.01397239,  0.00174077, -0.00803767,  0.03453857,  0.03404271,
        0.01145037, -0.00439764, -0.02802488,  0.00191273,  0.01188293],
      dtype=float32)

In [18]:
print("Vector for 'gripping' (first 10 dims):", model.wv["gripping"][:10])

Vector for 'gripping' (first 10 dims): [-0.0093056   0.01001705 -0.01113353  0.00610174  0.00811732  0.009136
  0.01068291  0.02038326 -0.03946117  0.0056218 ]
